<a href="https://colab.research.google.com/github/Mahathi-cs/course-registration-system/blob/main/SQL_Query_Generator_IBM_Granite_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Activity 1.1: Install all required Python packages
!pip install -q gradio transformers torch reportlab accelerate bitsandbytes black autopep8 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 24.2 MB/s eta 0:00:00


In [2]:
# Activity 1.2: Test model inference capabilities using pipeline
from transformers import pipeline
import torch

try:
    pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct", device_map="auto", torch_dtype=torch.float16)
    print("✅ Pipeline ready for testing!")
except Exception as e:
    print(f"❌ Pipeline error: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Pipeline ready for testing!


In [3]:
# Activity 1.2: Load Granite 3.3 2B Instruct model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "ibm-granite/granite-3.3-2b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16)

print("🚀 IBM Granite Model Loaded Successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

🚀 IBM Granite Model Loaded Successfully!


In [4]:
import gradio as gr
from datetime import datetime
from pathlib import Path
import sqlite3
import re
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse

# Activity 2.2: Implement Database Schema Definition
SCHEMA = {
    'users': {'id': 'INTEGER', 'name': 'TEXT', 'email': 'TEXT', 'signup_date': 'DATE', 'age': 'INTEGER', 'country': 'TEXT', 'status': 'TEXT'},
    'orders': {'id': 'INTEGER', 'user_id': 'INTEGER', 'product_name': 'TEXT', 'amount': 'REAL', 'order_date': 'DATE', 'status': 'TEXT'},
    'products': {'id': 'INTEGER', 'name': 'TEXT', 'price': 'REAL', 'category': 'TEXT', 'stock': 'INTEGER'},
    'transactions': {'id': 'INTEGER', 'user_id': 'INTEGER', 'amount': 'REAL', 'date': 'DATE', 'type': 'TEXT'}
}

In [5]:
# Activity 2.1: Build Security Validation System
DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE', 'EXEC', 'EXECUTE']
DANGEROUS_CHARS = [';', '--', '/*', '*/', 'xp_', 'sp_']

def is_safe(text):
    """Activity 2.1: Create is_safe() validation function"""
    text_upper = text.upper()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False
    for char in DANGEROUS_CHARS:
        if char in text:
            return False
    return True

In [6]:
# Activity 4.3: Functional Testing - Table detection logic
def get_table_name(query_text):
    """Intelligently infer table name from natural language"""
    query_lower = query_text.lower()

    keyword_map = {
        'users': ['user', 'users', 'customer', 'customers', 'account', 'accounts', 'profile'],
        'orders': ['order', 'orders', 'purchase', 'purchases', 'buy', 'bought'],
        'products': ['product', 'products', 'item', 'items', 'goods'],
        'transactions': ['transaction', 'transactions', 'payment', 'payments', 'transfer']
    }

    for table, keywords in keyword_map.items():
        for keyword in keywords:
            if keyword in query_lower:
                return table
    return 'users'  # Default table

In [7]:
# Activity 4.3: Column mapping and temporal condition extraction
def get_column_names(table_name, query_text):
    query_lower = query_text.lower()
    columns = list(SCHEMA[table_name].keys())
    selected_cols = []

    column_map = {
        'name': ['name', 'username', 'full name'],
        'email': ['email', 'mail', 'address'],
        'signup_date': ['signup', 'created', 'joined', 'registered', 'date', 'month'],
        'age': ['age', 'years'],
        'amount': ['amount', 'price', 'cost', 'total'],
        'date': ['date', 'when'],
        'status': ['status', 'state'],
        'country': ['country', 'location', 'place']
    }

    for col, keywords in column_map.items():
        if col in columns:
            for keyword in keywords:
                if keyword in query_lower:
                    selected_cols.append(col)
                    break
    return selected_cols if selected_cols else ['*']

def extract_conditions(query_text):
    query_lower = query_text.lower()
    conditions = []

    # Temporal conditions from your document
    if 'last month' in query_lower: conditions.append("DATE(signup_date) >= DATE('now', '-1 month')")
    if 'today' in query_lower: conditions.append("DATE(signup_date) = DATE('now')")
    if 'active' in query_lower: conditions.append("status = 'active'")

    # Amount filtering using Regex
    amount_match = re.search(r'more than|greater than|above\s+([0-9]+)', query_lower)
    if amount_match: conditions.append(f"amount > {amount_match.group(1)}")

    return conditions

In [8]:
query_history = []

def generate_sql(query_text):
    if not is_safe(query_text):
        return None, "❌ UNSAFE: Query contains dangerous SQL keywords!"

    table = get_table_name(query_text)
    columns = get_column_names(table, query_text)
    columns_str = ', '.join(columns)
    sql = f"SELECT {columns_str} FROM {table}"

    conditions = extract_conditions(query_text)
    if conditions: sql += " WHERE " + " AND ".join(conditions)
    if 'all' not in query_text.lower(): sql += " LIMIT 10"

    return sql, f"✅ Generated from table '{table}'"

def download_pdf():
    if not query_history: return "❌ No queries to export!", None
    filename = f"SQL_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    story = [Paragraph("🔍 SQL Query Report", styles['Title'])]
    for i, q in enumerate(query_history, 1):
        story.append(Paragraph(f"<b>Query #{i}:</b> {q['user']}", styles['Normal']))
        story.append(Paragraph(f"<code>{q['sql']}</code>", styles['Code']))
        story.append(Spacer(1, 12))
    doc.build(story)
    return f"✅ PDF Downloaded", filename

In [9]:
def process_query(user_query, chat_history):
    sql_query, explanation = generate_sql(user_query)
    response = f"✅ SQL:\n```sql\n{sql_query}\n```\n{explanation}"
    query_history.append({'user': user_query, 'sql': sql_query})
    chat_history.append((user_query, response))
    return chat_history

with gr.Blocks(theme=gr.themes.Soft(), title="SQL Generator") as demo:
    gr.Markdown("# 🔍 SQL Query Generator (IBM Granite 3.3)")
    chatbot = gr.Chatbot(label="Chat History")
    query_input = gr.Textbox(label="Ask in English (e.g., 'Users from last month')")

    with gr.Row():
        send_btn = gr.Button("Generate SQL", variant="primary")
        pdf_btn = gr.Button("📄 Download PDF")

    pdf_file = gr.File(label="Generated Report")

    send_btn.click(process_query, [query_input, chatbot], [chatbot])
    pdf_btn.click(download_pdf, None, [gr.Textbox(label="Status"), pdf_file])

demo.launch(share=True)

/tmp/ipykernel_501/481304408.py:8: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="SQL Generator") as demo:
/tmp/ipykernel_501/481304408.py:10: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History")
/tmp/ipykernel_501/481304408.py:10: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://685ceb382650f1a8d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Activity 3.3: Develop Export Functionality (TXT Export)
def download_txt():
    if not query_history:
        return "❌ No queries to export!", None
    content = f"SQL Query History Report\nGenerated: {datetime.now()}\n{'='*60}\n\n"
    for i, query in enumerate(query_history, 1):
        content += f"Query #{i}\nUser Request: {query['user']}\nGenerated SQL: {query['sql']}\n{'-'*60}\n\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"sql_queries_{timestamp}.txt"
    Path(filename).write_text(content)
    return f"✅ TXT Downloaded", filename

# Activity 3.4: Create Social Sharing Features
def generate_whatsapp_link():
    if not query_history: return "❌ No queries to share!"
    message = "🔍 *SQL Queries Generated*\n\n"
    for i, query in enumerate(query_history[-3:], 1): # Sharing last 3
        message += f"*Query {i}:* {query['user']}\n`{query['sql']}`\n\n"
    return f"✅ WhatsApp Link: https://wa.me/?text={urllib.parse.quote(message)}"

def generate_facebook_link():
    if not query_history: return "❌ No queries to share!"
    message = f"Check out my SQL Generator! Total queries: {len(query_history)}"
    return f"✅ FB Link: https://www.facebook.com/sharer/sharer.php?u=https://google.com&quote={urllib.parse.quote(message)}"

# Activity 3.5: Apply Custom Styling & Build UI
professional_css = """
body { background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); color: white; }
.gradio-container { border-radius: 20px; box-shadow: 0 10px 30px rgba(0,0,0,0.5); }
.button-primary { background: linear-gradient(135deg, #FF6B6B 0%, #FF8E72 100%) !important; border: none !important; }
.button-secondary { background: linear-gradient(135deg, #4ECDC4 0%, #44A08D 100%) !important; border: none !important; }
"""

with gr.Blocks(theme=gr.themes.Base(), css=professional_css, title="SQL Generator Pro") as demo:
    gr.Markdown("# 🔍 SQL Query Generator\n### Natural Language → SQLite SQL (IBM Granite 3.3)")

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Query Conversion History", height=400)
            query_input = gr.Textbox(label="📝 Describe your data request", placeholder="e.g., Show me users from India who signed up last month")
            send_btn = gr.Button("Generate SQL", variant="primary")

        with gr.Column(scale=1):
            gr.Markdown("### 📥 Export & Share")
            pdf_btn = gr.Button("📄 Download PDF", variant="secondary")
            txt_btn = gr.Button("📋 Download TXT", variant="secondary")
            wa_btn = gr.Button("💬 Share on WhatsApp")
            fb_btn = gr.Button("👍 Share on Facebook")

            output_file = gr.File(label="Download Center")
            status_box = gr.Textbox(label="Status", interactive=False)

    # Activity 4.1: End-to-End Integration
    send_btn.click(process_query, [query_input, chatbot], [chatbot])
    pdf_btn.click(download_pdf, None, [status_box, output_file])
    txt_btn.click(download_txt, None, [status_box, output_file])
    wa_btn.click(generate_whatsapp_link, None, status_box)
    fb_btn.click(generate_facebook_link, None, status_box)

# Activity 5.1: Deployment
demo.launch(share=True, debug=True)

/tmp/ipykernel_501/649967039.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=professional_css, title="SQL Generator Pro") as demo:
/tmp/ipykernel_501/649967039.py:35: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), css=professional_css, title="SQL Generator Pro") as demo:
/tmp/ipykernel_501/649967039.py:40: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Query Conversion History", height=400)
/tmp/ipyke

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://17007f7003a8d9cc1a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
